# U2T01 — Adapting BERT for NLP tasks (Colab / Kaggle runner)

This notebook runs **the same modules as the repository**, on a GPU instead of a local Mac.
Nothing is reimplemented here: `src/` is the single source of truth, so a run started on a
T4 and a run started on an M-series Mac produce the same `results/<task>/<run_id>.json`
schema and the same report.

Runtime → Change runtime type → **T4 GPU** before running anything.


## 1. Get the code


In [ ]:
# Option A — clone the repository (set it to your fork/remote):
GITHUB_REPO = ''  # e.g. 'https://github.com/<user>/adapting-bert-nlp-tasks.git'

import os, pathlib
if GITHUB_REPO:
    !git clone -q $GITHUB_REPO u2t01
    os.chdir('u2t01')
else:
    # Option B — upload the project as a zip (Files pane, or the cell below).
    from google.colab import files  # type: ignore
    up = files.upload()
    !unzip -q -o "{list(up)[0]}" -d .
    os.chdir([p for p in pathlib.Path('.').iterdir() if (p / 'src').is_dir()][0])
print('working directory:', pathlib.Path.cwd())


## 2. Dependencies

Colab already ships a CUDA build of torch; installing the pinned CPU/MPS wheel would
replace it, so torch is filtered out and everything else is installed at the pinned version.


In [ ]:
!grep -v '^torch==' requirements.txt > /tmp/req_colab.txt
!pip install -q -r /tmp/req_colab.txt

import torch, transformers, datasets
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('transformers', transformers.__version__, '| datasets', datasets.__version__)


## 3. Sanity check the subword/label alignment

Run this **before** any training. A silent misalignment still produces a loss curve that
looks perfectly healthy while the model learns garbage.


In [ ]:
!python scripts/sanity_check.py | head -60


## 4. Run the experiment grid

Resumable: a run whose `results/<task>/<run_id>.json` exists is skipped, which matters on
Colab because the session can be recycled. Drop `--task ...` to run everything.

On a T4 the whole grid is roughly 1–1.5 h; the SQuAD runs dominate it.


In [ ]:
!python scripts/run_experiments.py --task agnews ner pos


In [ ]:
!python scripts/run_experiments.py --task qa


### Optional: the DistilBERT size benchmark

`bert-base` is the delivered body; DistilBERT (~66 M) and BERT-large (~340 M) are reported
only as size benchmarks. BERT-large at sequence length 384 will OOM on a T4 — lower the
batch size in `configs/experiments.py` if you want it.


In [ ]:
!python scripts/run_experiments.py --optional


## 5. Figures, tables and the report


In [ ]:
!python scripts/make_figures.py
!python scripts/build_report.py --no-pdf   # Chrome is not available on Colab

from IPython.display import Image, display
for name in ('ladder', 'cost_benefit', 'curves'):
    display(Image(f'report/figures/{name}.png'))


In [ ]:
import json, glob
from src.report_data import rows
for r in rows():
    if r['headline'] is not None:
        print(f"{r['task']:7s} {r['method']:38s} {r['headline_name']:15s} "
              f"{r['headline']:6.2f}  trainable={r['trainable']:>11,}  {r['minutes']:5.1f} min")


## 6. Take the results home

`results/` holds every metric, loss curve and timing; `models/` holds the checkpoints that
`scripts/push_to_hub.py` publishes.


In [ ]:
!zip -qr u2t01_results.zip results report/figures docs
from google.colab import files  # type: ignore
files.download('u2t01_results.zip')


## 7. Publish to the HuggingFace Hub


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
!python scripts/push_to_hub.py --user <your-hf-username>
